# 02 · Audit the current synthetic ConsentShield dataset

**Important:** the supplied dataset has already been generated by the current Python data-preparation scripts. This notebook audits those artifacts; it intentionally does **not** regenerate or overwrite them with the older notebook's incompatible schema.

**Inputs:** `baseline_payloads.jsonl`, `drift_payloads.jsonl`, `policy_labels.jsonl`, the three `field_classification_*.csv` files, three synthetic text splits, `adversarial_cases.jsonl`, `dataset_manifest.json`, and `dataset_quality_report.json`.

**Output:** `results/synthetic_dataset_audit.json`.

The audit checks file counts, label distribution, split/group isolation, payload-to-field-path consistency, and policy-label integrity. It does not prove that the synthetic patterns fully represent real traffic.

In [ ]:
import json, csv, collections, platform
from pathlib import Path

HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / "data" / "synthetic").is_dir()), HERE)
DATA = ROOT / "data"
SYN = DATA / "synthetic"
RESULTS = ROOT / "results"; RESULTS.mkdir(parents=True, exist_ok=True)
required = [SYN / "baseline_payloads.jsonl", SYN / "drift_payloads.jsonl", SYN / "policy_labels.jsonl"]
missing = [str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError("Missing required dataset files: " + ", ".join(missing))

def read_jsonl(path):
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def flatten_values(obj, prefix=""):
    out = {}
    if isinstance(obj, dict):
        for key, value in obj.items():
            path = f"{prefix}.{key}" if prefix else str(key)
            out.update(flatten_values(value, path))
    elif isinstance(obj, list):
        for value in obj:
            out.update(flatten_values(value, prefix + "[]"))
    else:
        out[prefix] = obj
    return out

baseline = read_jsonl(SYN / "baseline_payloads.jsonl")
drift = read_jsonl(SYN / "drift_payloads.jsonl")
policies = read_jsonl(SYN / "policy_labels.jsonl")
payloads = baseline + drift
payload_by_id = {r["record_id"]: r for r in payloads}
policy_by_id = {r["record_id"]: r for r in policies}
print("Machine:", platform.machine())
print("Payloads:", len(payloads), "baseline:", len(baseline), "drift:", len(drift))
print("Policy labels:", len(policies))

## Payload and policy integrity

A clean payload is expected to be allowed; policy labels are checked for ID coverage, duplicate IDs, forwarding consistency, and whether listed sensitive paths exist in the original payload. For this dataset's intended labels, `expected_downstream_contains_sensitive` should be false for all records after the expected action.

In [ ]:
scenario_counts = collections.Counter(r.get("scenario", "UNKNOWN") for r in payloads)
action_counts = collections.Counter(r.get("expected_action", "UNKNOWN") for r in policies)
all_ids = [r.get("record_id") for r in payloads]
policy_ids = [r.get("record_id") for r in policies]
issues = []
if len(all_ids) != len(set(all_ids)): issues.append("duplicate payload record_id")
if len(policy_ids) != len(set(policy_ids)): issues.append("duplicate policy record_id")
if set(all_ids) != set(policy_ids): issues.append("payload/policy record_id sets differ")
for rec_id, p in policy_by_id.items():
    payload_row = payload_by_id.get(rec_id)
    if not payload_row: continue
    flat = flatten_values(payload_row["payload"])
    for path in p.get("sensitive_fields", []):
        if path not in flat: issues.append(f"{rec_id}: sensitive path absent: {path}")
    for path in p.get("expected_redacted_fields", []):
        if path not in flat: issues.append(f"{rec_id}: redacted path absent: {path}")
    for path in p.get("expected_removed_fields", []):
        if path not in flat: issues.append(f"{rec_id}: removed path absent: {path}")
    if p.get("expected_downstream_contains_sensitive") is not False:
        issues.append(f"{rec_id}: expected downstream sensitive flag is not false")
    action = p.get("expected_action")
    forwarded = bool(p.get("expected_forwarded"))
    if action == "QUARANTINE_PAYLOAD" and forwarded:
        issues.append(f"{rec_id}: payload quarantine marked forwarded")
    if action == "ALLOW" and not forwarded:
        issues.append(f"{rec_id}: ALLOW marked not forwarded")

# Record split isolation: each record/group should belong to one split.
split_by_id = {r["record_id"]: r.get("split") for r in payloads}
group_splits = collections.defaultdict(set)
for r in payloads: group_splits[r.get("group_id", r["record_id"])].add(r.get("split"))
field_reports = {}
for split in ("train", "validation", "test"):
    path = SYN / f"field_classification_{split}.csv"
    if not path.exists():
        issues.append(f"missing {path.name}"); continue
    rows = list(csv.DictReader(path.open(encoding="utf-8", newline="")))
    labels = collections.Counter(int(r["label_sensitive"]) for r in rows)
    drift_labels = collections.Counter(int(r["label_drift_anomaly"]) for r in rows)
    field_reports[split] = {"rows": len(rows), "sensitive_positive": labels[1], "sensitive_negative": labels[0],
                            "drift_positive": drift_labels[1], "drift_negative": drift_labels[0],
                            "positive_rate": labels[1] / len(rows) if rows else 0.0}
    bad_record_refs = sum(r["record_id"] not in payload_by_id for r in rows)
    if bad_record_refs: issues.append(f"{split}: {bad_record_refs} field rows reference missing payload IDs")

report = {"machine": platform.machine(), "payload_count": len(payloads), "baseline_count": len(baseline),
          "drift_count": len(drift), "scenario_counts": dict(scenario_counts),
          "policy_label_count": len(policies), "action_counts": dict(action_counts),
          "field_classification": field_reports, "policy_issues": issues,
          "cross_split_group_count": sum(len(v) > 1 for v in group_splits.values()),
          "manifest": json.loads((SYN / "dataset_manifest.json").read_text()) if (SYN / "dataset_manifest.json").exists() else None,
          "quality_report": json.loads((SYN / "dataset_quality_report.json").read_text()) if (SYN / "dataset_quality_report.json").exists() else None}
print("Scenario counts:", dict(scenario_counts))
print("Action counts:", dict(action_counts))
print("Field split summary:", json.dumps(field_reports, indent=2))
print("Cross-split groups:", report["cross_split_group_count"])
print("Integrity issues:", len(issues))
for issue in issues[:30]: print(" -", issue)
if not issues: print("No policy/payload integrity issues found by these checks.")

In [ ]:
# Audit synthetic text spans and ensure group IDs do not cross splits.
text_summary = {}
text_group_splits = collections.defaultdict(set)
for split in ("train", "validation", "test"):
    path = SYN / f"consentshield_text_{split}.jsonl"
    if not path.exists():
        text_summary[split] = {"missing": True}; continue
    rows = read_jsonl(path)
    label_counts = collections.Counter()
    positive = 0; invalid = 0
    for r in rows:
        tokens = r.get("tokenized_text") or []
        spans = r.get("ner") or []
        positive += int(bool(spans))
        for s, e, lab in spans:
            label_counts[lab] += 1
            if not (0 <= int(s) <= int(e) < len(tokens)): invalid += 1
        text_group_splits[r.get("group_id", r["id"])].add(split)
    text_summary[split] = {"rows": len(rows), "positive_rows": positive, "negative_rows": len(rows)-positive,
                           "positive_rate": positive/len(rows) if rows else 0,
                           "span_counts": dict(label_counts), "invalid_spans": invalid}
report["synthetic_text"] = text_summary
report["text_cross_split_group_count"] = sum(len(v) > 1 for v in text_group_splits.values())
print(json.dumps(text_summary, indent=2))
print("Text groups appearing in multiple splits:", report["text_cross_split_group_count"])
(RESULTS / "synthetic_dataset_audit.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print("Saved", RESULTS / "synthetic_dataset_audit.json")